In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
df = pd.read_parquet("data/mp_total/results.parquet")
flat = df.reset_index()

In [ ]:
# Best F1 and MCC per benchmark
for bench in df.index.get_level_values("benchmark").unique():
    group = df.loc[bench]
    best_f1 = group.loc[group["f1_score"].idxmax()]
    best_mcc = group.loc[group["mcc"].idxmax()]
    print(
        f"{bench:25s} "
        f"F1={best_f1['f1_score']:.4f} (k={best_f1['max_iterations']})  "
        f"MCC={best_mcc['mcc']:.4f} (k={best_mcc['max_iterations']})"
    )

In [ ]:
# Full results table sorted by F1
flat.sort_values("f1_score", ascending=False)[
    [
        "benchmark",
        "max_iterations",
        "sae_l0",
        "true_l0",
        "precision",
        "recall",
        "f1_score",
        "mcc",
        "explained_variance",
    ]
].head(12)

## F1 & MCC vs max_iterations across distributions

In [ ]:
melted = flat.melt(
    id_vars=["benchmark", "max_iterations"],
    value_vars=["f1_score", "mcc"],
    var_name="metric",
    value_name="score",
)

fig = px.line(
    melted,
    x="max_iterations",
    y="score",
    color="benchmark",
    facet_col="metric",
    category_orders={"metric": ["f1_score", "mcc"]},
    markers=True,
    labels={
        "max_iterations": "Max Iterations",
        "score": "Score",
        "benchmark": "Benchmark",
    },
    title="F1 & MCC vs Max Iterations by Distribution",
    height=500,
    width=1100,
)
fig.show()

## Precision & Recall vs max_iterations across distributions

In [ ]:
melted = flat.melt(
    id_vars=["benchmark", "max_iterations"],
    value_vars=["precision", "recall"],
    var_name="metric",
    value_name="score",
)

fig = px.line(
    melted,
    x="max_iterations",
    y="score",
    color="benchmark",
    facet_col="metric",
    category_orders={"metric": ["precision", "recall"]},
    markers=True,
    labels={
        "max_iterations": "Max Iterations",
        "score": "Score",
        "benchmark": "Benchmark",
    },
    title="Precision & Recall vs Max Iterations by Distribution",
    height=500,
    width=1100,
)
fig.show()

## F1 heatmap: benchmark × max_iterations

In [ ]:
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["F1 Score", "MCC"],
)

for i, metric in enumerate(["f1_score", "mcc"]):
    pivot = flat.pivot_table(values=metric, index="benchmark", columns="max_iterations")
    fig.add_trace(
        go.Heatmap(
            z=pivot.values,
            x=[str(c) for c in pivot.columns],
            y=pivot.index.tolist(),
            colorscale="Viridis",
            showscale=(i == 1),
            text=pivot.values.round(3),
            texttemplate="%{text}",
        ),
        row=1,
        col=i + 1,
    )

fig.update_layout(
    height=400,
    width=1100,
    title_text="Benchmark × Max Iterations",
)
fig.update_xaxes(title_text="max_iterations")
fig.show()

## sae_l0 vs true_l0 across distributions

In [ ]:
fig = px.scatter(
    flat,
    x="true_l0",
    y="sae_l0",
    color="benchmark",
    symbol="max_iterations",
    hover_data=["f1_score", "mcc"],
    labels={
        "true_l0": "True L0",
        "sae_l0": "SAE L0",
        "benchmark": "Benchmark",
        "max_iterations": "Max Iter",
    },
    title="SAE L0 vs True L0 (each point = one config)",
    height=500,
    width=800,
)
# Add y=x reference line
max_val = max(flat["true_l0"].max(), flat["sae_l0"].max())
fig.add_shape(
    type="line",
    x0=0,
    y0=0,
    x1=max_val,
    y1=max_val,
    line=dict(dash="dash", color="gray"),
)
fig.show()